# 02. 피처 엔지니어링

EDA에서 얻은 시사점으로 초기 사이클(2–100)에서 **셀 단위 피처**를 만들고, 피처 세트를 정의한다.
한 행 = 한 배터리 셀 (타깃이 셀당 1개이므로 사이클이 아니라 셀이 분석 단위).

| 구분 | 피처 | 근거 |
|---|---|---|
| ΔQ(V) 통계 | `log10 Var`, `log10 \|min\|`, `log10 \|mean\|`, 첨도, 왜도, 2V 지점 | Q3: 전압 구간별 용량 변화가 장·단수명을 가른다 |
| Summary | QD 기울기(2–100, 91–100), 초기 용량, IR, 평균 온도(Tmax/Tavg), 충전시간 | Q2·Q4·Q5 |
| 충전 프로토콜 | 1단계 C-rate, 전환 SOC, 2단계 C-rate, 평균 C-rate | Q4 |

**ΔQ(V) = Q₁₀₀(V) − Q₁₀(V)** 는 전압축으로 보간된 방전 용량 `Qdlin`(2.0–3.5V, 1,000점)에서 계산한다
(배열 인덱스 = cycle−1 이므로 cycle 10·100 = 인덱스 9·99; Batch 1의 cycle 1은 비어 있다).

In [1]:
import sys, pathlib, warnings, io, runpy, contextlib
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from src.config import R, FIG_DIR
from src.data import build_dataset, training_pool, test_set, FEATURE_SETS, FEATURE_SET_DOC

def run_eda(module):
    """EDA 스크립트를 실행해 results/figures 와 results/eda 를 (재)생성한다 (출력은 숨김)."""
    with contextlib.redirect_stdout(io.StringIO()):
        runpy.run_module(module, run_name="__main__")


In [2]:
ds = build_dataset()
pool = training_pool(ds); test = test_set(ds)
print(f"학습 풀 {len(pool)}셀 / {pool.protocol.nunique()}개 프로토콜,  테스트(B2) {len(test)}셀")
cols = ["batch", "cell_id", "protocol", "cycle_life", "log_dq_var", "log_dq_min", "dq_kurt", "qd_2", "qd_slope_91_100", "tmax_mean", "switch", "avg_c"]
display(pool[cols].head(6).round(3))

학습 풀 36셀 / 20개 프로토콜,  테스트(B2) 39셀


,batch,cell_id,protocol,cycle_life,log_dq_var,log_dq_min,dq_kurt,qd_2,qd_slope_91_100,tmax_mean,switch,avg_c
0,batch1,5,4.4C(80%)-4.4C,1074.0,-4.179,-1.599,-1.228,1.076,-0.0,33.020,0.8,4.400
1,batch1,6,4.8C(80%)-4.8C,636.0,-3.769,-1.422,-1.238,1.076,-0.0,35.793,0.8,4.800
2,batch1,7,4.8C(80%)-4.8C,870.0,-3.813,-1.418,-1.121,1.094,-0.0,35.323,0.8,4.800
3,batch1,9,5.4C(40%)-3.6C,1054.0,-4.059,-1.542,-1.182,1.083,-0.0,35.985,0.4,4.320
4,batch1,11,5.4C(50%)-3C,788.0,-4.147,-1.625,-1.206,1.054,-0.0,36.232,0.5,4.154
5,batch1,14,5.4C(60%)-3C,880.0,-4.046,-1.548,-1.313,1.071,-0.0,36.631,0.6,4.500


## 피처 세트

| 세트 | 구성 | 설명 |
|---|---|---|
| **A** | `log Var[ΔQ]` | 핵심 피처 1개 |
| **B** | ΔQ 통계 3종 | Var / min / mean (서로 상관 0.98–0.99 → 규제로 처리) |
| **C** | A + QD 기울기(91–100), 전환 SOC, Tmax, 첨도, 초기 용량 | Day 1 EDA에서 고른 6개 (B2·B3 EDA를 참고한 한계) |
| **D** | 전체 후보 15개 | 참고용 (다중공선성 확인) |
| **E** | **학습 데이터 내부**에서 프로토콜 서브샘플링으로 부호가 안정적인 피처만 선별 | Day 1의 한계(B2·B3를 참고한 선별)를 보완 — 매 학습 분할 안에서 새로 선별 |

In [3]:
display(pd.DataFrame({"피처 수": {k: len(v) for k, v in FEATURE_SETS.items()}, "설명": FEATURE_SET_DOC}).loc[["A", "B", "C", "D"]])
print("E 는 학습 데이터에서 선별:", FEATURE_SET_DOC["E"])

,피처 수,설명
A,1.0,Var[ΔQ] 1개
B,3.0,ΔQ 통계 3종
C,6.0,Day 1 선별 6개(B2·B3 EDA 참고)
D,15.0,전체 후보 15개


E 는 학습 데이터에서 선별: B1 내부 안정성 선별(프로토콜 서브샘플링)


## 세트 E: 학습 데이터(B1) 내부 안정성 선별

프로토콜의 80%를 50번 반복 추출해 Spearman ρ를 계산하고, |평균 ρ| ≥ 0.4이면서 부호 일치율 ≥ 0.95인 피처를 남긴 뒤
서로 상관이 0.9 이상인 중복 피처를 제거한다. Hold-out/CV의 각 학습 부분에서만 호출하므로 검증 셀이 선별에 섞이지 않는다.

In [4]:
from src.feature_selection import stable_features, vif
chosen, tab = stable_features(pool)
display(tab.sort_values("mean_rho", key=abs, ascending=False).round(3).head(10))
print("선별(전체 학습 풀 기준):", chosen)

,feature,mean_rho,std_rho,sign_agree
0,log_dq_var,-0.806,0.037,1.0
1,log_dq_min,-0.768,0.046,1.0
2,log_dq_abs_mean,-0.738,0.056,1.0
5,dq_2v,0.626,0.075,1.0
8,qd_slope_2_100,0.444,0.077,1.0
3,dq_skew,-0.407,0.094,1.0
13,chargetime_mean,0.404,0.083,1.0
10,qd_100,0.372,0.085,1.0
9,qd_slope_91_100,0.287,0.107,1.0
7,qd_max_minus_2,0.255,0.102,1.0


선별(전체 학습 풀 기준): ['log_dq_var', 'dq_2v', 'qd_slope_2_100', 'dq_skew', 'chargetime_mean']


> **관찰** B1 내부에서는 `chargetime_mean`(ρ≈+0.40) 같은 프로토콜 계열 피처도 부호가 안정적이다. 하지만 Day 1 EDA에서 이 피처는 **B2에서 부호가 뒤집힌다**(+0.60 → −0.80).
> 즉 학습 데이터 안의 안정성만으로는 "배치가 바뀌어도 통하는 피처"를 가려낼 수 없다 — 03번 노트북의 핵심 발견과 이어진다.

## 다중공선성 (VIF)

VIF가 10을 넘으면 계수 추정이 불안정하다.

In [5]:
out = {}
for k in ["A", "B", "C", "D"]:
    v = vif(pool[FEATURE_SETS[k]]) if len(FEATURE_SETS[k]) > 1 else pd.Series({FEATURE_SETS[k][0]: 1.0})
    out[k] = round(float(v.max()), 1)
display(pd.Series(out, name="세트별 최대 VIF").to_frame().T)
display(vif(pool[FEATURE_SETS["B"]]).round(1).to_frame("VIF (세트 B)"))

,A,B,C,D
세트별 최대 VIF,1.0,415.3,2.2,1838.4


,VIF (세트 B)
log_dq_min,415.3
log_dq_var,208.5
log_dq_abs_mean,72.3


세트 B의 ΔQ 통계 3종은 서로 거의 같은 정보라 VIF가 매우 크다 → **일반 OLS보다 규제 모델(Elastic Net)** 을 후보에 넣는다.
세트 C는 VIF가 모두 3 미만으로 안전하다.

**전처리 원칙** (모든 변환·대체는 학습 데이터에서만 계산, `src/models.py`의 파이프라인 안에서 수행)
- 피처·타깃 모두 log10 변환 후 표준화, IR 측정 실패(결측 6셀)는 학습 데이터 중앙값으로 대체
- 모델 예측은 10^ŷ로 역변환해 원 스케일 MAPE로 평가